In [96]:
import re

JD = """
Senior Data Analyst
Company: Labcorp
Location: Mumbai

Responsibilities:
a) Build dashboards in Power BI and present insights to stakeholders.
b) Work with Adobe Customer Journey Analytics (CJA) to analyse campaign data.
Requirements:
1) 3+ years of experience in marketing analytics.
2) Strong SQL and Python. Experience with Marketo is a plus.
Bachelor's degree in Engineering or related field.
"""

def clean_line(line):
    line = line.replace("’", "'").replace("–", "-").replace("—", "-")
    line = re.sub(r"^\s*(\(?[a-zA-Z0-9]{1,2}[\)\.]|[-•*])\s+", "", line)
    line = re.sub(r"\s+", " ", line).strip()
    return line


lines = [clean_line(l) for l in JD.splitlines()]
lines = [l for l in lines if l]

sentences = []
for l in lines:
    parts = re.split(r"(?<=[.!?])\s+(?=[A-Z])", l)
    sentences.extend(p.strip() for p in parts if p.strip())

for i, s in enumerate(sentences):
    print(i, "|", s)

0 | Senior Data Analyst
1 | Company: Labcorp
2 | Location: Mumbai
3 | Responsibilities:
4 | Build dashboards in Power BI and present insights to stakeholders.
5 | Work with Adobe Customer Journey Analytics (CJA) to analyse campaign data.
6 | Requirements:
7 | 3+ years of experience in marketing analytics.
8 | Strong SQL and Python.
9 | Experience with Marketo is a plus.
10 | Bachelor's degree in Engineering or related field.


In [98]:
SECTION_WORDS = {
    "responsibilities": ["responsibilit", "what you will do", "duties"],
    "requirements":     ["requirement", "qualification", "must have", "what you bring"],
    "preferred":        ["preferred", "nice to have", "good to have", "bonus"],
    "about":            ["about"],
}

def heading_of(line):
    low = line.lower().rstrip(":").strip()
    if not line.endswith(":") or len(line.split()) > 6:
        return None
    for name, words in SECTION_WORDS.items():
        if any(w in low for w in words):
            return name
    return None

meta = {}
rows = []                    # (section, sentence)
section = "header"
for s in sentences:
    h = heading_of(s)
    if h:
        section = h
        continue
    m = re.match(r"^(Company|Location):\s*(.+)$", s)
    if m:
        meta[m.group(1).lower()] = m.group(2)
        continue
    rows.append((section, s))

print("META:", meta)
for sec, s in rows:
    print(f"{sec:16} | {s}")

META: {'company': 'Labcorp', 'location': 'Mumbai'}
header           | Senior Data Analyst
responsibilities | Build dashboards in Power BI and present insights to stakeholders.
responsibilities | Work with Adobe Customer Journey Analytics (CJA) to analyse campaign data.
requirements     | 3+ years of experience in marketing analytics.
requirements     | Strong SQL and Python.
requirements     | Experience with Marketo is a plus.
requirements     | Bachelor's degree in Engineering or related field.


In [100]:
from transformers import AutoTokenizer, AutoModelForTokenClassification
import torch

NAME = "jjzha/jobbert_knowledge_extraction"
tok = AutoTokenizer.from_pretrained(NAME)
model = AutoModelForTokenClassification.from_pretrained(NAME)
model.eval()
print("labels:", model.config.id2label)

def raw_tags(sentence):
    enc = tok(sentence, return_tensors="pt", truncation=True, max_length=256)
    with torch.no_grad():
        logits = model(**enc).logits[0]
    probs = logits.softmax(-1)
    ids = probs.argmax(-1).tolist()
    pieces = tok.convert_ids_to_tokens(enc["input_ids"][0])
    return [(p, model.config.id2label[i], round(probs[n, i].item(), 2))
            for n, (p, i) in enumerate(zip(pieces, ids))]

for sec, s in rows[:5]:
    print("\n", s)
    for piece, label, conf in raw_tags(s):
        print(f"  {piece:15} {label:3} {conf}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

labels: {0: 'B', 1: 'I', 2: 'O'}

 Senior Data Analyst
  [CLS]           O   1.0
  Senior          O   1.0
  Data            O   1.0
  Ana             O   1.0
  ##ly            O   1.0
  ##st            O   1.0
  [SEP]           O   1.0

 Build dashboards in Power BI and present insights to stakeholders.
  [CLS]           O   1.0
  B               O   1.0
  ##uild          O   1.0
  dash            O   1.0
  ##boards        O   1.0
  in              O   1.0
  Power           B   1.0
  B               I   1.0
  ##I             I   0.96
  and             O   1.0
  present         O   1.0
  insights        O   1.0
  to              O   1.0
  stakeholders    O   1.0
  .               O   1.0
  [SEP]           O   1.0

 Work with Adobe Customer Journey Analytics (CJA) to analyse campaign data.
  [CLS]           O   1.0
  Work            O   1.0
  with            O   1.0
  Adobe           O   1.0
  Custom          O   1.0
  ##er            O   1.0
  Journey         O   1.0
  Ana             

In [101]:
SKILL_NAME = "jjzha/jobbert_skill_extraction"
tok_s = AutoTokenizer.from_pretrained(SKILL_NAME)
model_s = AutoModelForTokenClassification.from_pretrained(SKILL_NAME).eval()

def word_tags(sentence, tk, md):
    enc = tk(sentence, return_tensors="pt", truncation=True, max_length=256)
    with torch.no_grad():
        probs = md(**enc).logits[0].softmax(-1)
    ids = probs.argmax(-1).tolist()
    pieces = tk.convert_ids_to_tokens(enc["input_ids"][0])
    words = []                                   # [word, label, conf]
    for n, (p, i) in enumerate(zip(pieces, ids)):
        if p in ("[CLS]", "[SEP]"):
            continue
        label, conf = md.config.id2label[i], probs[n, i].item()
        if p.startswith("##") and words:
            words[-1][0] += p[2:]                # extend word, keep first piece's label
        else:
            words.append([p, label, conf])
    return words

def to_spans(words):
    spans, cur = [], []
    for w, label, conf in words:
        if label == "B":
            if cur: spans.append(" ".join(cur))
            cur = [w]
        elif label == "I" and cur:
            cur.append(w)
        else:                                    # O, or stray I with nothing open
            if cur: spans.append(" ".join(cur))
            cur = []
    if cur: spans.append(" ".join(cur))
    return spans

for sec, s in rows:
    k = to_spans(word_tags(s, tok, model))
    sk = to_spans(word_tags(s, tok_s, model_s))
    print(f"\n{s}\n  knowledge model: {k}\n  skill model    : {sk}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]


Senior Data Analyst
  knowledge model: []
  skill model    : []

Build dashboards in Power BI and present insights to stakeholders.
  knowledge model: ['Power BI']
  skill model    : ['Build dashboards', 'present insights to stakeholders']

Work with Adobe Customer Journey Analytics (CJA) to analyse campaign data.
  knowledge model: []
  skill model    : ['analyse campaign data']

3+ years of experience in marketing analytics.
  knowledge model: ['marketing analytics']
  skill model    : []

Strong SQL and Python.
  knowledge model: ['SQL', 'Python']
  skill model    : []

Experience with Marketo is a plus.
  knowledge model: ['Marketo']
  skill model    : []

Bachelor's degree in Engineering or related field.
  knowledge model: ['Bachelor', "'", 's degree in Engineering']
  skill model    : []


In [102]:
DEGREE = re.compile(r"\b(bachelor|master|phd|degree|b\.?tech|m\.?tech)", re.I)
PLUS_CUE = re.compile(r"\b(plus|preferred|nice to have|bonus|good to have|desirable)\b", re.I)
KNOWN_TOOLS = {"sql", "python", "power bi", "tableau", "excel", "marketo",
               "powerpoint", "pyspark", "spark", "aws", "azure", "r"}

def capitalised_phrases(text):
    return re.findall(
        r"(?:(?<=\s)|^)((?:[A-Z][A-Za-z0-9&+#.\-]*)(?:\s+[A-Z][A-Za-z0-9&+#.\-]*)+)", text)

def acronym_aliases(sentence, phrases):
    alias = {}
    for a in re.findall(r"\(([A-Z]{2,6})\)", sentence):
        for p in phrases:
            w = p.split()
            if len(w) >= len(a) and "".join(x[0] for x in w[-len(a):]) == a:
                alias[p] = a
    return alias

def extract(section, sentence):
    s = re.sub(r"(\w)'s\b", r"\1s", sentence)
    items = [(t, "knowledge") for t in to_spans(word_tags(s, tok, model))]
    items += [(t, "skill") for t in to_spans(word_tags(s, tok_s, model_s))]
    alias = {}
    if section != "header":
        body = s.split(" ", 1)[1] if " " in s else ""
        phrases = capitalised_phrases(body)
        alias = acronym_aliases(s, phrases)
        for p in phrases:
            if not any(p.lower() in t.lower() or t.lower() in p.lower() for t, _ in items):
                items.append((p, "rule"))
    nice = bool(PLUS_CUE.search(s)) or section == "preferred"
    out = []
    for t, src in items:
        if DEGREE.search(t):
            continue
        if src == "knowledge":
            kind = "tool" if t.lower() in KNOWN_TOOLS else "domain"
        elif src == "rule":
            kind = "tool"
        else:
            kind = "task"
        out.append(dict(text=t, source=src, kind=kind, section=section,
                        nice=nice, alias=alias.get(t)))
    return out

found = []
for sec, s in rows:
    found += extract(sec, s)

print(f"{'text':36} {'source':10} {'kind':7} {'section':17} nice  alias")
for f in found:
    print(f"{f['text']:36} {f['source']:10} {f['kind']:7} {f['section']:17} {str(f['nice']):5} {f['alias'] or ''}")

text                                 source     kind    section           nice  alias
Power BI                             knowledge  tool    responsibilities  False 
Build dashboards                     skill      task    responsibilities  False 
present insights to stakeholders     skill      task    responsibilities  False 
analyse campaign data                skill      task    responsibilities  False 
Adobe Customer Journey Analytics     rule       tool    responsibilities  False CJA
marketing analytics                  knowledge  domain  requirements      False 
SQL                                  knowledge  tool    requirements      False 
Python                               knowledge  tool    requirements      False 
Marketo                              knowledge  tool    requirements      True  


In [103]:
import pdfplumber

RESUME_PDF = r"C:\Users\Zahid Shaikh\z\Hiriq\Zahid_Salim_Shaikh__Copy_.pdf"   # <- your path

def glue_share(text):
    words = [w for w in text.split()
             if not ("://" in w or "@" in w or w.lower().startswith("www."))]
    total = sum(len(w) for w in words)
    return sum(len(w) for w in words if len(w) >= 25) / total if total >= 40 else 0.0

with pdfplumber.open(RESUME_PDF) as pdf:
    page = pdf.pages[0]
    default = page.extract_text() or ""
    tight   = page.extract_text(x_tolerance=1.5) or ""

print("pages:", len(pdf.pages))
print(f"default (3.0)  glue share: {glue_share(default):.3f}")
print(f"tight   (1.5)  glue share: {glue_share(tight):.3f}")
print("\n--- default, first 400 chars ---\n", default[:400])
print("\n--- tight, first 400 chars ---\n", tight[:400])

pages: 2
default (3.0)  glue share: 0.929
tight   (1.5)  glue share: 0.000

--- default, first 400 chars ---
 Zahid Salim Shaikh
Mumbai,India | zahidshaikh10101@gmail.com | +91-8286092787 | Portfolio | LinkedIn | Github
M.TechinDataScience&EngineeringfromBITSPilaniwith4+yearsofexperiencebuildingscalabledata
productsandanalyticssystemsthatdrivemeasurablebusinessimpact. ExpertinPython,SQL,andPySpark-
shippedGenAI-poweredclassificationpipelinesinproduction,builtself-serveanalyticsplatformsreducinginsight
lat

--- tight, first 400 chars ---
 Zahid Salim Shaikh
Mumbai, India | zahidshaikh10101@gmail.com | +91-8286092787 | Portfolio | LinkedIn | Github
M.Tech in Data Science & Engineering from BITS Pilani with 4+ years of experience building scalable data
products and analytics systems that drive measurable business impact. Expert in Python, SQL, and PySpark -
shipped GenAI-powered classification pipelines in production, built self-serv


In [104]:
RESUME_HEADS = {"skills": ["skill", "technical", "tools", "technologies"],
                "experience": ["experience", "employment", "work history"],
                "education": ["education", "academic"],
                "projects": ["project"],
                "certifications": ["certif", "course"],
                "summary": ["summary", "profile", "objective"]}

def resume_heading(line):
    low = line.lower().rstrip(":").strip()
    if len(line.split()) > 4:
        return None
    for name, words in RESUME_HEADS.items():
        if any(w in low for w in words):
            return name
    return None

with pdfplumber.open(RESUME_PDF) as pdf:
    r_text = "\n".join((p.extract_text(x_tolerance=1.5) or "") for p in pdf.pages)

r_lines = [clean_line(l) for l in r_text.splitlines()]
r_lines = [l for l in r_lines if l]

listed, found_r = {}, []
section = "header"
for line in r_lines:
    h = resume_heading(line)
    if h:
        section = h
        continue
    if section == "header" and ("@" in line or "|" in line):
        continue                                     # contact details
    if section == "skills":
        body = line.split(":", 1)[1] if ":" in line else line
        for item in re.split(r"[,;|/]", body):
            item = item.strip(" .")
            if 1 < len(item) < 40:
                listed[item.lower()] = item
        continue
    for sent in re.split(r"(?<=[.!?])\s+(?=[A-Z])", line):
        if len(sent.split()) >= 3:
            found_r += extract(section, sent)

print("LISTED in Skills section:", len(listed))
print(sorted(listed.values(), key=str.lower))

seen = {}
for f in found_r:
    seen.setdefault(f["text"].lower(), f)
for kind in ("tool", "domain", "task"):
    print(f"\n{kind.upper()} found in text:")
    print(sorted({f['text'] for f in seen.values() if f['kind'] == kind}, key=str.lower))

LISTED in Skills section: 49
['Agile', 'Analysis', 'B Testing', 'Business Intelligence', 'ClickUp', 'Code', 'Cohort Analysis', 'Data Automation', 'Data Cleaning', 'Data Modelling', 'Data Pipeline', 'Databricks', 'EDA', 'Excel', 'FastAPI', 'Feature Engineering', 'Generative AI', 'GitHub', 'JioSaavn)', 'JIRA', 'Jupyter Notebook', 'LLM (Hugging Face', 'Looker', 'Matplotlib', 'Model Validation', 'NumPy', 'OpenRouter)', 'Pandas', 'Plotly', 'Postman', 'Power BI', 'Prompt Engineering', 'PySpark', 'Python', 'REST APIs (Spotify', 'Root Cause Analysis', 'Scikit-learn', 'Scrum', 'Seaborn', 'SQL', 'Stakeholder Management', 'Statistical', 'Streamlit', 'Supervised & Unsupervised Learning', 'TensorFlow', 'Trend Analysis', 'VS', 'YouTube', 'YouTube CMS']

TOOL found in text:
['College Portal Verify', 'Cover Letter Generator', 'Data Science Verify', 'Development Intern', 'DSP APIs', 'Executive Business Intelligence', 'Innovative Research', 'International Journal', 'IVY Entertainment Pvt.', 'Mumbai Apr'

In [105]:
def spans_conf(words):
    spans, cur, conf = [], [], []
    def close():
        if cur:
            spans.append((" ".join(cur), min(conf)))
    for w, label, c in words:
        if label == "B":
            close(); cur, conf = [w], [c]
        elif label == "I" and cur:
            cur.append(w); conf.append(c)
        else:
            close(); cur, conf = [], []
    close()
    return spans

JUNK = {"year", "years", "end", "back", "front", "multi", "system", "interactive", "producing", "data"}

def good_span(t):
    t = t.strip(" ,-/()")
    return len(t) >= 2 and re.search(r"[A-Za-z]", t) is not None and t.lower() not in JUNK

def extract2(section, sentence, rescue=True, min_conf=0.8):
    s = re.sub(r"(\w)'s\b", r"\1s", sentence)
    s = re.sub(r"(?<=[A-Za-z])-(?=[A-Za-z])", " ", s)          # front-end -> front end
    items = [(t, "knowledge", c) for t, c in spans_conf(word_tags(s, tok, model))]
    items += [(t, "skill", c) for t, c in spans_conf(word_tags(s, tok_s, model_s))]
    alias = {}
    if rescue and section != "header":
        body = s.split(" ", 1)[1] if " " in s else ""
        phrases = capitalised_phrases(body)
        alias = acronym_aliases(s, phrases)
        for p in phrases:
            if not any(p.lower() in t.lower() or t.lower() in p.lower() for t, _, _ in items):
                items.append((p, "rule", 1.0))
    nice = bool(PLUS_CUE.search(s)) or section == "preferred"
    out = []
    for t, src, c in items:
        t = t.strip(" ,-/()")
        if DEGREE.search(t) or not good_span(t):
            continue
        if src == "knowledge" and t.lower() not in KNOWN_TOOLS and c < min_conf:
            continue
        kind = ("tool" if t.lower() in KNOWN_TOOLS else "domain") if src == "knowledge" \
               else "tool" if src == "rule" else "task"
        out.append(dict(text=t, source=src, kind=kind, section=section,
                        nice=nice, alias=alias.get(t), conf=round(c, 2)))
    return out

def split_skill_line(line):
    body = line.split(":", 1)[1] if ":" in line else line
    inner = re.findall(r"\(([^)]*)\)", body)
    body = re.sub(r"\([^)]*\)", "", body)
    parts = re.split(r"[,;|]", body + "," + ",".join(inner))
    return [p.strip(" .") for p in parts if 1 < len(p.strip(" .")) < 40]

listed, found_r = {}, []
section = "header"
for line in r_lines:
    h = resume_heading(line)
    if h:
        section = h
        continue
    if section == "header" and ("@" in line or "|" in line):
        continue
    if section == "skills":
        for item in split_skill_line(line):
            listed[item.lower()] = item
        continue
    for sent in re.split(r"(?<=[.!?])\s+(?=[A-Z])", line):
        if len(sent.split()) >= 3:
            found_r += extract2(section, sent, rescue=False)

print("LISTED:", len(listed))
print(sorted(listed.values(), key=str.lower))
seen = {}
for f in found_r:
    seen.setdefault(f["text"].lower(), f)
for kind in ("tool", "domain"):
    print(f"\n{kind.upper()} found in text (not already listed):")
    print(sorted({f['text'] for f in seen.values()
                  if f['kind'] == kind and f['text'].lower() not in listed}, key=str.lower))

LISTED: 50
['A/B Testing', 'Agile / Scrum', 'Analysis', 'Business Intelligence', 'ClickUp', 'Code', 'Cohort Analysis', 'Data Automation', 'Data Cleaning', 'Data Modelling', 'Data Pipeline', 'Databricks', 'EDA', 'Excel', 'FastAPI', 'Feature Engineering', 'Generative AI', 'GitHub', 'Hugging Face', 'JioSaavn', 'JIRA', 'Jupyter Notebook', 'LLM', 'Looker', 'Matplotlib', 'Model Validation', 'NumPy', 'OpenRouter', 'Pandas', 'Plotly', 'Postman', 'Power BI', 'Prompt Engineering', 'PySpark', 'Python', 'REST APIs', 'Root Cause Analysis', 'Scikit-learn', 'Seaborn', 'Spotify', 'SQL', 'Stakeholder Management', 'Statistical', 'Streamlit', 'Supervised & Unsupervised Learning', 'TensorFlow', 'Trend Analysis', 'VS', 'YouTube', 'YouTube CMS']

TOOL found in text (not already listed):
[]

DOMAIN found in text (not already listed):
['Agile release cycles', 'APPLY', 'automated signal monitoring', 'automation', 'B testing', 'BI dashboards', 'BITS', 'BORDERLINE', 'CD', 'CI', 'CSS', 'Data Science & Engineering

In [106]:
# 1) rebuild listed skills with wrapped lines joined
sk_lines, sec = [], "header"
for line in r_lines:
    h = resume_heading(line)
    if h:
        sec = h
        continue
    if sec == "skills":
        if sk_lines and ":" not in line:
            sk_lines[-1] += " " + line          # wrapped continuation
        else:
            sk_lines.append(line)
listed = {}
for line in sk_lines:
    for item in split_skill_line(line):
        listed[item.lower()] = item
print("listed skills now:", len(listed))
print([x for x in listed.values() if x.lower() in ("statistical analysis", "vs code", "vs", "code", "analysis")])

# 2) the job's skills (rescue rule ON, models ON)
jd_found = []
for sec_, s_ in rows:
    jd_found += extract2(sec_, s_, rescue=True)

def norm(t):
    return re.sub(r"[^a-z0-9+#]+", " ", t.lower()).strip()

jd_skills = {}
for f in jd_found:
    if f["kind"] == "task":
        continue
    jd_skills.setdefault(norm(f["text"]), f)

# 3) evidence from the resume
resume_norm = " " + norm(r_text) + " "
evidence = {norm(x) for x in listed.values()} | {norm(f["text"]) for f in found_r}

def match_one(skill, alias=None):
    for name in filter(None, [skill, alias]):
        n = norm(name)
        if n in evidence:
            return "listed/found", 1.0
        if f" {n} " in resume_norm:
            return "text mention", 1.0
    return "missing", 0.0

WEIGHT = {"tool": 1.0, "domain": 0.4}
num = den = 0.0
print(f"\n{'job skill':36} {'kind':7} {'nice':5} {'w':5} {'evidence':14} credit")
for f in jd_skills.values():
    w = WEIGHT[f["kind"]] * (0.5 if f["nice"] else 1.0)
    how, credit = match_one(f["text"], f["alias"])
    num += w * credit
    den += w
    print(f"{f['text']:36} {f['kind']:7} {str(f['nice']):5} {w:<5} {how:14} {credit}")
print(f"\nSKILLS SCORE = {num:.2f} / {den:.2f} = {100 * num / den:.0f}%")

listed skills now: 48
['Statistical Analysis', 'VS Code']

job skill                            kind    nice  w     evidence       credit
Power BI                             tool    False 1.0   listed/found   1.0
Adobe Customer Journey Analytics     tool    False 1.0   missing        0.0
marketing analytics                  domain  False 0.4   missing        0.0
SQL                                  tool    False 1.0   listed/found   1.0
Python                               tool    False 1.0   listed/found   1.0
Marketo                              tool    True  0.5   missing        0.0

SKILLS SCORE = 3.00 / 4.90 = 61%


In [107]:
from sentence_transformers import SentenceTransformer, util

mini = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

# candidates = things the resume really says, each with a real line as proof
cands = {}
for v in listed.values():
    cands.setdefault(v, f"Skills section: {v}")
for f in found_r:
    t = f["text"]
    if t not in cands:
        proof = next((l for l in r_lines if t in l), None)
        if proof:
            cands[t] = proof
cand_texts = list(cands)
cand_emb = mini.encode(cand_texts, convert_to_tensor=True, normalize_embeddings=True)
print("resume candidates:", len(cand_texts))

def semantic(skill):
    e = mini.encode(skill, convert_to_tensor=True, normalize_embeddings=True)
    sims = util.cos_sim(e, cand_emb)[0]
    top = sims.topk(3)
    best = [(cand_texts[i], round(float(v), 2)) for v, i in zip(top.values, top.indices)]
    s = best[0][1]
    credit = 0.5 if s >= 0.65 else 0.25 if s >= 0.55 else 0.0
    return best, credit

num = den = 0.0
print(f"\n{'job skill':34} {'w':5} {'evidence':14} credit")
for f in jd_skills.values():
    w = WEIGHT[f["kind"]] * (0.5 if f["nice"] else 1.0)
    how, credit = match_one(f["text"], f["alias"])
    best = None
    if how == "missing":
        best, credit = semantic(f["text"])
        how = "meaning" if credit > 0 else "missing"
    num += w * credit
    den += w
    print(f"{f['text']:34} {w:<5} {how:14} {credit}")
    if best:
        for t, sc in best:
            print(f"      near: {t:30} {sc}   <- {cands[t][:70]}")
print(f"\nSKILLS SCORE (with meaning) = {num:.2f} / {den:.2f} = {100 * num / den:.0f}%")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

resume candidates: 119

job skill                          w     evidence       credit
Power BI                           1.0   listed/found   1.0
Adobe Customer Journey Analytics   1.0   missing        0.0
      near: Google Data Analytics          0.51   <- Google Data Analytics Verify
      near: Business Intelligence          0.43   <- Skills section: Business Intelligence
      near: streaming market data pipelines 0.37   <- strategy dashboards, and automated signal monitoring using streaming m
marketing analytics                0.4   meaning        0.5
      near: Google Data Analytics          0.71   <- Google Data Analytics Verify
      near: Business Intelligence          0.6   <- Skills section: Business Intelligence
      near: Statistical Analysis           0.5   <- Skills section: Statistical Analysis
SQL                                1.0   listed/found   1.0
Python                             1.0   listed/found   1.0
Marketo                            0.5   missing      

In [112]:
LEGAL = re.compile(r"equal opportunity|privacy statement|accessibility|reasonable accommodation|"
                   r"protected veteran|applicable law|background check|without regard to|affirmative action", re.I)
FILLER = {"certification", "reporting", "tools", "tool", "skills", "skill", "dashboards", "dashboard",
          "experience", "platform", "framework", "frameworks"}

def dedupe_key(t):
    return " ".join(w for w in norm(t).split() if w not in FILLER)

def pipeline_skills(text):
    text = re.sub(r"\b(Role|Responsibilities|Requirements|Qualifications|Skills|Offer|Benefits)(?=[A-Z][a-z])",
                  r"\1\n", text)
    lines = [clean_line(l) for l in text.splitlines()]
    sents = []
    for l in lines:
        if l:
            sents += [p.strip() for p in re.split(r"(?<=[.!?])\s+(?=[A-Z])", l) if p.strip()]
    section, found = "body", []
    for s in sents:
        h = heading_of(s)
        if h:
            section = h
            continue
        if LEGAL.search(s):
            continue
        found += extract2(section, s, rescue=True)
    best = {}
    for f in found:
        if f["kind"] == "task":
            continue
        k = dedupe_key(f["text"])
        if k and (k not in best or len(f["text"]) < len(best[k]["text"])):
            best[k] = f
    return best

def evaluate(posts, only_checked=False):
    tp_ex = ex = gold_found = gold_n = 0
    for p in posts:
        if only_checked and not p["checked"]:
            continue
        pred = [f["text"] for f in pipeline_skills(p["description"]).values()]
        gold = p["skills"]
        tp = [t for t in pred if any(same(t, g) for g in gold)]
        fp = [t for t in pred if t not in tp]
        fn = [g for g in gold if not any(same(t, g) for t in pred)]
        tp_ex += len(tp); ex += len(pred); gold_found += len(gold) - len(fn); gold_n += len(gold)
        print(f"\n{p['key'][:60]}   checked={p['checked']}")
        print(f"  extracted {len(pred)}  gold {len(gold)}  "
              f"precision {len(tp)/max(len(pred),1):.2f}  recall {(len(gold)-len(fn))/max(len(gold),1):.2f}")
        print("  WRONG :", fp)
        print("  MISSED:", fn)
    print(f"\nOVERALL  precision {tp_ex/max(ex,1):.2f}   recall {gold_found/max(gold_n,1):.2f}   "
          f"({ex} extracted, {gold_n} gold)")

evaluate(posts)


Senior Data Analyst - AI & Automation | QualityAI   checked=False
  extracted 37  gold 18  precision 0.54  recall 1.00
  WRONG : ['AI', 'data analysis', 'Data Analyst', 'scripting', 'AI First Automation', 'reconciliation', 'Technical Skills', 'business intelligence', 'Programming', 'data extraction', 'manipulation', 'pipeline creation', 'Google Ecosystem', 'Modern Frameworks', 'dynamic context sharing', 'BI', 'MindsetAnalytical Thinking']
  MISSED: []

Senior Analyst, Digital Analytics | Labcorp   checked=False
  extracted 26  gold 10  precision 0.42  recall 1.00
  WRONG : ['Senior Analyst', 'Digital Analytics', 'LabCorp Bangalore', 'Senior Associate', 'Campaign Performance Reporting', 'data analysis', 'automation', 'CJA', 'Adobe CJA', 'workspace reporting', 'analysis', 'website traffic', 'campaigns', 'channels', 'engagement']
  MISSED: []

OVERALL  precision 0.49   recall 1.00   (63 extracted, 28 gold)


In [113]:
def show(i):
    p = posts[i]
    text = re.sub(r"\b(Role|Responsibilities|Requirements|Qualifications|Skills|Offer|Benefits)(?=[A-Z][a-z])",
                  r"\1\n", p["description"])
    print(f"[{i}] {p['key']}   checked={p['checked']}\n")
    n = 0
    for l in text.splitlines():
        l = clean_line(l)
        for s in re.split(r"(?<=[.!?])\s+(?=[A-Z])", l):
            if s.strip():
                print(f"{n:3} | {s.strip()}")
                n += 1

def set_gold(i, skills_csv, split=None):
    skills = [x.strip().lower() for x in skills_csv.split(",") if x.strip()]
    posts[i]["skills"] = skills
    posts[i]["checked"] = True
    if split:
        posts[i]["split"] = split
    json.dump(posts, open(LABELS, "w", encoding="utf-8"), indent=2, ensure_ascii=False)
    print(f"saved {len(skills)} skills for post {i}")

show(0)

[0] Senior Data Analyst - AI & Automation | QualityAI   checked=False

  0 | About the Role
  1 | We are looking for an innovative Data Analyst with AI Experience to bridge traditional data analysis with modern AI-driven automation.
  2 | You will manage large datasets, build insightful dashboards, and leverage cutting-edge AI and scripting tools to fundamentally transform how data workflows operate within a Google-integrated ecosystem.Key Responsibilities
  3 | Data Analysis & Dashboards: Process complex datasets using SQL, Python, and Google Sheets, and build interactive reports using Tableau, Power BI, or Looker Studio.AI-First Automation & MCP Integration: Design and deploy Model Context Protocol (MCP) standards and LLM integrations to allow AI agents to safely connect with enterprise data sources and execute multi-step workflows.Advanced Scripting: Build robust data pipelines and complex automations using Python, and streamline day-to-day operations using JavaScript and Google App

In [114]:
def split_sentences(text):
    text = re.sub(r"\b(Role|Responsibilities|Requirements|Qualifications|Skills|Offer|Benefits)(?=[A-Z][a-z])",
                  r"\1\n", text)
    text = re.sub(r"(?<=[a-z\)])([.!?])(?=[A-Z])", r"\1\n", text)       # "Studio.AI-First" -> two sentences
    out = []
    for l in text.splitlines():
        l = clean_line(l)
        out += [p.strip() for p in re.split(r"(?<=[.!?])\s+(?=[A-Z])", l) if p.strip()]
    return out

def pipeline_skills(text):
    section, found = "body", []
    for s in split_sentences(text):
        h = heading_of(s)
        if h:
            section = h
            continue
        if LEGAL.search(s):
            continue
        found += extract2(section, s, rescue=True)
    best = {}
    for f in found:
        if f["kind"] == "task":
            continue
        k = dedupe_key(f["text"])
        if k and (k not in best or len(f["text"]) < len(best[k]["text"])):
            best[k] = f
    return best

def show(i):
    p = posts[i]
    print(f"[{i}] {p['key']}   checked={p['checked']}\n")
    for n, s in enumerate(split_sentences(p["description"])):
        print(f"{n:3} | {s}")

show(0)

[0] Senior Data Analyst - AI & Automation | QualityAI   checked=False

  0 | About the Role
  1 | We are looking for an innovative Data Analyst with AI Experience to bridge traditional data analysis with modern AI-driven automation.
  2 | You will manage large datasets, build insightful dashboards, and leverage cutting-edge AI and scripting tools to fundamentally transform how data workflows operate within a Google-integrated ecosystem.
  3 | Key Responsibilities
  4 | Data Analysis & Dashboards: Process complex datasets using SQL, Python, and Google Sheets, and build interactive reports using Tableau, Power BI, or Looker Studio.
  5 | AI-First Automation & MCP Integration: Design and deploy Model Context Protocol (MCP) standards and LLM integrations to allow AI agents to safely connect with enterprise data sources and execute multi-step workflows.
  6 | Advanced Scripting: Build robust data pipelines and complex automations using Python, and streamline day-to-day operations using Java

In [115]:
SECTION_WORDS = {
    "responsibilities": ["responsibilit", "what you will do", "duties"],
    "requirements":     ["requirement", "qualification", "must have", "what you bring", "skills"],
    "preferred":        ["preferred", "nice to have", "good to have", "bonus"],
    "about":            ["about"],
}

def heading_of(line):
    line = line.strip()
    low = line.lower().rstrip(":").strip()
    if len(line.split()) > 4 or line.endswith((".", "!", "?")):
        return None
    for name, words in SECTION_WORDS.items():
        if any(w in low for w in words):
            return name
    return None

for i, s in enumerate(split_sentences(posts[0]["description"])):
    print(i, heading_of(s) or "-", "|", s[:60])

0 about | About the Role
1 - | We are looking for an innovative Data Analyst with AI Experi
2 - | You will manage large datasets, build insightful dashboards,
3 responsibilities | Key Responsibilities
4 - | Data Analysis & Dashboards: Process complex datasets using S
5 - | AI-First Automation & MCP Integration: Design and deploy Mod
6 - | Advanced Scripting: Build robust data pipelines and complex 
7 - | ETL & Data Governance: Manage basic ETL frameworks, data val
8 - | Stakeholder Collaboration: Translate complex technical findi
9 requirements | Candidate Requirements
10 requirements | Core Technical Skills
11 - | Experience: 5 to 8+ years of professional experience in a da
12 - | Programming & Databases: Solid foundation in Python and SQL 
13 - | Google Ecosystem & AI Tooling: Hands-on experience with Goog
14 - | Modern Frameworks: Familiarity with Model Context Protocol (
15 - | BI & Visualization: Expertise in data visualization platform
16 - | Soft Skills & MindsetAnalytical Think

In [117]:
# %% [markdown]
# # Hiriq: JobBERT skill matching, built step by step
# Run the cells top to bottom. Change the three paths in cell 0, then Run All.
#
# | Cell | What it does |
# |---|---|
# | 0 | Settings and imports |
# | 1 | Clean text and split into sentences |
# | 2 | Find sections (responsibilities, requirements, ...) |
# | 3 | Load JobBERT (two models), read raw tags, merge pieces into spans |
# | 4 | Filters, rescue rule, groups (tool / domain / task) |
# | 5 | Skills from a job post (pipeline_skills) |
# | 6 | Read the resume PDF (glue fix, skills list, bullets) |
# | 7 | Match: exact, text mention, meaning (MiniLM), skills score |
# | 8 | Hand labels: show, set_gold, evaluate (precision / recall) |

# %% 0. Settings and imports
# pip install transformers torch sentence-transformers pdfplumber
import re, json, os

RESUME_PDF = r"C:\Users\Zahid Shaikh\z\Hiriq\Zahid_Salim_Shaikh__Copy_.pdf"
LABELS     = r"C:\Users\Zahid Shaikh\z\Hiriq\hiriq\hiriq\ml\notebooks\samples\labels_draft.json"

DEMO_JD = """
Description
Job Descriptions
Company: Trent
Location: Mumbai

Key Responsibilities:

Assist in developing and implementing machine learning models to address business
Help build and maintain data pipelines and reporting systems for business metrics
Conduct exploratory data analysis to uncover patterns and insights
Prepare and clean datasets for analysis and model building
Create visualizations to communicate analytical findings effectively
Identify customer behavior patterns to inform business strategy
Qualifications/Experience:

Programming skills in Python, with knowledge of data science libraries (NumPy, Pandas, Scikit-learn, etc.)
Experience with SQL for data extraction and manipulation, Familiarity with dashboard creation using PowerBI or similar tools
Understanding of fundamental statistical techniques and machine learning algorithms, Knowledge of cloud platforms (preferably Azure)
Knowledge of machine learning approaches such as Regression, Classification techniques, Clustering Neural Networks, CV, NLP etc., Familiarity with version control systems (Git) and collaborative workflows, Ability to communicate technical concepts to non-technical audiences
Strong analytical thinking and problem-solving skills, Eagerness to learn and grow in the field of data science
Competencies
Functional/ Technical Skills*

Interpersonal/ Communication Skills*

Professional Impression*

Relevant Background/Special Skill Set*
"""

# %% 1. Clean text and split into sentences
HEADING_GLUE = r"\b(Role|Responsibilities|Requirements|Qualifications|Skills|Offer|Benefits)(?=[A-Z][a-z])"

def clean_line(line):
    line = line.replace("\u2019", "'").replace("\u2013", "-").replace("\u2014", "-")
    line = re.sub(r"^\s*(\(?[a-zA-Z0-9]{1,2}[\)\.]|[-\u2022*])\s+", "", line)   # strip a) 1) - bullet
    return re.sub(r"\s+", " ", line).strip()

def split_sentences(text):
    text = re.sub(HEADING_GLUE, r"\1\n", text)                       # "ResponsibilitiesData" -> two lines
    text = re.sub(r"(?<=[a-z\)])([.!?])(?=[A-Z])", r"\1\n", text)   # "Studio.AI-First" -> two sentences
    out = []
    for l in text.splitlines():
        l = clean_line(l)
        out += [p.strip() for p in re.split(r"(?<=[.!?])\s+(?=[A-Z])", l) if p.strip()]
    return out

# %% 2. Sections
SECTION_WORDS = {
    "responsibilities": ["responsibilit", "what you will do", "duties"],
    "requirements":     ["requirement", "qualification", "must have", "what you bring", "skills"],
    "preferred":        ["preferred", "nice to have", "good to have", "bonus"],
    "about":            ["about"],
}

def heading_of(line):
    """A heading is a short line (4 words or fewer, no full stop) containing a heading word, with or without a colon."""
    line = line.strip()
    low = line.lower().rstrip(":").strip()
    if len(line.split()) > 4 or line.endswith((".", "!", "?")):
        return None
    for name, words in SECTION_WORDS.items():
        if any(w in low for w in words):
            return name
    return None

# %% 3. JobBERT: raw tags -> words -> spans
import torch
from transformers import AutoTokenizer, AutoModelForTokenClassification

KNOWLEDGE_MODEL = "jjzha/jobbert_knowledge_extraction"   # tools and domains
SKILL_MODEL     = "jjzha/jobbert_skill_extraction"       # duties and abilities

tok   = AutoTokenizer.from_pretrained(KNOWLEDGE_MODEL)
model = AutoModelForTokenClassification.from_pretrained(KNOWLEDGE_MODEL).eval()
tok_s   = AutoTokenizer.from_pretrained(SKILL_MODEL)
model_s = AutoModelForTokenClassification.from_pretrained(SKILL_MODEL).eval()
print("labels:", model.config.id2label)          # B = begin, I = inside, O = outside

def word_tags(sentence, tk, md):
    """[[word, label, confidence], ...]. Word pieces (##) extend the word and the word keeps its first piece's label."""
    enc = tk(sentence, return_tensors="pt", truncation=True, max_length=256)
    with torch.no_grad():
        probs = md(**enc).logits[0].softmax(-1)
    ids = probs.argmax(-1).tolist()
    pieces = tk.convert_ids_to_tokens(enc["input_ids"][0])
    words = []
    for n, (p, i) in enumerate(zip(pieces, ids)):
        if p in ("[CLS]", "[SEP]"):
            continue
        label, conf = md.config.id2label[i], probs[n, i].item()
        if p.startswith("##") and words:
            words[-1][0] += p[2:]
        else:
            words.append([p, label, conf])
    return words

def spans_conf(words):
    """B starts a span, I continues it, O closes it. Each span carries its lowest word confidence."""
    spans, cur, conf = [], [], []
    def close():
        if cur:
            spans.append((" ".join(cur), min(conf)))
    for w, label, c in words:
        if label == "B":
            close(); cur, conf = [w], [c]
        elif label == "I" and cur:
            cur.append(w); conf.append(c)
        else:
            close(); cur, conf = [], []
    close()
    return spans

# %% 4. Filters, rescue rule, groups
DEGREE   = re.compile(r"\b(bachelor|master|phd|degree|b\.?tech|m\.?tech)", re.I)
PLUS_CUE = re.compile(r"\b(plus|preferred|nice to have|bonus|good to have|desirable)\b", re.I)
LEGAL    = re.compile(r"equal opportunity|privacy statement|accessibility|reasonable accommodation|"
                      r"protected veteran|applicable law|background check|without regard to|affirmative action", re.I)
KNOWN_TOOLS = {"sql", "python", "power bi", "tableau", "excel", "marketo",
               "powerpoint", "pyspark", "spark", "aws", "azure", "r"}
JUNK   = {"year", "years", "end", "back", "front", "multi", "system", "interactive", "producing", "data"}
FILLER = {"certification", "reporting", "tools", "tool", "skills", "skill", "dashboards", "dashboard",
          "experience", "platform", "framework", "frameworks"}
WEIGHT = {"tool": 1.0, "domain": 0.4}          # a "plus" skill counts half again (placeholder, calibrated later)

def norm(t):
    return re.sub(r"[^a-z0-9+#]+", " ", t.lower()).strip()

def dedupe_key(t):
    return " ".join(w for w in norm(t).split() if w not in FILLER)

def good_span(t):
    t = t.strip(" ,-/()")
    return len(t) >= 2 and re.search(r"[A-Za-z]", t) is not None and t.lower() not in JUNK

def capitalised_phrases(text):
    return re.findall(
        r"(?:(?<=\s)|^)((?:[A-Z][A-Za-z0-9&+#.\-]*)(?:\s+[A-Z][A-Za-z0-9&+#.\-]*)+)", text)

def acronym_aliases(sentence, phrases):
    alias = {}
    for a in re.findall(r"\(([A-Z]{2,6})\)", sentence):
        for p in phrases:
            w = p.split()
            if len(w) >= len(a) and "".join(x[0] for x in w[-len(a):]) == a:
                alias[p] = a
    return alias

def extract2(section, sentence, rescue=True, min_conf=0.8):
    """Skills in one sentence: [{text, source, kind, section, nice, alias, conf}]"""
    s = re.sub(r"(\w)'s\b", r"\1s", sentence)                 # Bachelor's -> Bachelors
    s = re.sub(r"(?<=[A-Za-z])-(?=[A-Za-z])", " ", s)         # front-end -> front end
    items  = [(t, "knowledge", c) for t, c in spans_conf(word_tags(s, tok, model))]
    items += [(t, "skill", c)     for t, c in spans_conf(word_tags(s, tok_s, model_s))]
    alias = {}
    if rescue and section != "header":                         # models miss rare product names
        body = s.split(" ", 1)[1] if " " in s else ""          # skip the sentence's first word
        phrases = capitalised_phrases(body)
        alias = acronym_aliases(s, phrases)
        for p in phrases:
            if not any(p.lower() in t.lower() or t.lower() in p.lower() for t, _, _ in items):
                items.append((p, "rule", 1.0))
    nice = bool(PLUS_CUE.search(s)) or section == "preferred"
    out = []
    for t, src, c in items:
        t = t.strip(" ,-/()")
        if DEGREE.search(t) or not good_span(t):
            continue
        if src == "knowledge" and t.lower() not in KNOWN_TOOLS and c < min_conf:
            continue                                           # unknown name needs confidence
        if src == "knowledge":
            kind = "tool" if t.lower() in KNOWN_TOOLS else "domain"
        else:
            kind = "tool" if src == "rule" else "task"
        out.append(dict(text=t, source=src, kind=kind, section=section,
                        nice=nice, alias=alias.get(t), conf=round(c, 2)))
    return out

# %% 5. Skills from a job post
def pipeline_skills(text, init_section="body"):
    """{dedupe_key: skill dict} for one job post. Tasks are left out."""
    section, found = init_section, []
    for s in split_sentences(text):
        h = heading_of(s)
        if h:
            section = h
            continue
        if re.match(r"^(Company|Location):", s) or LEGAL.search(s):
            continue
        found += extract2(section, s, rescue=True)
    best = {}
    for f in found:
        if f["kind"] == "task":
            continue
        k = dedupe_key(f["text"])
        if k and (k not in best or len(f["text"]) < len(best[k]["text"])):
            best[k] = f
    return best

jd_skills = pipeline_skills(DEMO_JD, init_section="header")
print(f"{'skill':36} {'source':10} {'kind':7} {'section':17} nice  alias")
for f in jd_skills.values():
    print(f"{f['text']:36} {f['source']:10} {f['kind']:7} {f['section']:17} {str(f['nice']):5} {f['alias'] or ''}")

# %% 6. Read the resume
import pdfplumber

RESUME_HEADS = {"skills": ["skill", "technical", "tools", "technologies"],
                "experience": ["experience", "employment", "work history"],
                "education": ["education", "academic"],
                "projects": ["project"],
                "certifications": ["certif", "course"],
                "summary": ["summary", "profile", "objective"]}

def resume_heading(line):
    low = line.lower().rstrip(":").strip()
    if len(line.split()) > 4:
        return None
    for name, words in RESUME_HEADS.items():
        if any(w in low for w in words):
            return name
    return None

def glue_share(text):
    """Share of letters in 25+ character runs with no space. About 0 for real text, most of the page when glued."""
    words = [w for w in text.split() if not ("://" in w or "@" in w or w.lower().startswith("www."))]
    total = sum(len(w) for w in words)
    return sum(len(w) for w in words if len(w) >= 25) / total if total >= 40 else 0.0

def pdf_text(path):
    out = []
    with pdfplumber.open(path) as pdf:
        for page in pdf.pages:
            default = page.extract_text() or ""
            if glue_share(default) > 0.15:                    # LaTeX PDFs: words run together at the 3pt default
                tight = page.extract_text(x_tolerance=1.5) or ""
                if tight.strip() and glue_share(tight) < glue_share(default):
                    default = tight
            out.append(default)
    return "\n".join(out)

def split_skill_line(line):
    body = line.split(":", 1)[1] if ":" in line else line
    inner = re.findall(r"\(([^)]*)\)", body)                  # LLM (Hugging Face, OpenRouter)
    body = re.sub(r"\([^)]*\)", "", body)
    parts = re.split(r"[,;|]", body + "," + ",".join(inner))
    return [p.strip(" .") for p in parts if 1 < len(p.strip(" .")) < 40]

def read_resume(path):
    """(raw text, cleaned lines, listed skills, skills found in bullets)"""
    text = pdf_text(path)
    lines = [clean_line(l) for l in text.splitlines()]
    lines = [l for l in lines if l]
    sk_lines, section = [], "header"
    for line in lines:                                        # skills section: join wrapped lines
        h = resume_heading(line)
        if h:
            section = h
            continue
        if section == "skills":
            if sk_lines and ":" not in line:
                sk_lines[-1] += " " + line
            else:
                sk_lines.append(line)
    listed = {}
    for line in sk_lines:
        for item in split_skill_line(line):
            listed[item.lower()] = item
    found, section = [], "header"
    for line in lines:                                        # bullets and summary: models, no rescue rule
        h = resume_heading(line)
        if h:
            section = h
            continue
        if section == "skills" or (section == "header" and ("@" in line or "|" in line)):
            continue
        for sent in re.split(r"(?<=[.!?])\s+(?=[A-Z])", line):
            if len(sent.split()) >= 3:
                found += extract2(section, sent, rescue=False)
    return text, lines, listed, found

if os.path.exists(RESUME_PDF):
    r_text, r_lines, listed, found_r = read_resume(RESUME_PDF)
    print("glue share (default read):", round(glue_share(pdfplumber.open(RESUME_PDF).pages[0].extract_text() or ""), 3))
    print("listed skills:", len(listed))
    print(sorted(listed.values(), key=str.lower))
    extra = {f["text"] for f in found_r if f["kind"] != "task" and f["text"].lower() not in listed}
    print("\nfound in text, not listed:", sorted(extra, key=str.lower))
else:
    print("RESUME_PDF not found, set the path in cell 0")

# %% 7. Match the job against the resume
from sentence_transformers import SentenceTransformer, util
mini = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

def build_evidence():
    resume_norm = " " + norm(r_text) + " "
    evidence = {norm(x) for x in listed.values()} | {norm(f["text"]) for f in found_r}
    cands = {v: f"Skills section: {v}" for v in listed.values()}
    for f in found_r:
        t = f["text"]
        if t not in cands:
            proof = next((l for l in r_lines if t in l), None)
            if proof:
                cands[t] = proof
    texts = list(cands)
    emb = mini.encode(texts, convert_to_tensor=True, normalize_embeddings=True)
    return resume_norm, evidence, cands, texts, emb

def score_skills(jd, show_near=True):
    resume_norm, evidence, cands, texts, emb = build_evidence()

    def exact(skill, alias):
        for name in filter(None, [skill, alias]):
            n = norm(name)
            if n in evidence:
                return "listed/found", 1.0
            if f" {n} " in resume_norm:
                return "text mention", 1.0
        return "missing", 0.0

    def semantic(skill):
        e = mini.encode(skill, convert_to_tensor=True, normalize_embeddings=True)
        top = util.cos_sim(e, emb)[0].topk(3)
        best = [(texts[i], round(float(v), 2)) for v, i in zip(top.values, top.indices)]
        s = best[0][1]
        return best, (0.5 if s >= 0.65 else 0.25 if s >= 0.55 else 0.0)   # meaning-only credit is capped

    num = den = 0.0
    print(f"{'job skill':34} {'w':5} {'evidence':14} credit")
    for f in jd.values():
        w = WEIGHT[f["kind"]] * (0.5 if f["nice"] else 1.0)
        how, credit = exact(f["text"], f["alias"])
        best = None
        if how == "missing":
            best, credit = semantic(f["text"])
            how = "meaning" if credit > 0 else "missing"
        num += w * credit
        den += w
        print(f"{f['text']:34} {w:<5} {how:14} {credit}")
        if best and show_near:
            for t, sc in best:
                print(f"      near: {t:30} {sc}   <- {cands[t][:70]}")
    pct = 100 * num / den if den else 0.0
    print(f"\nSKILLS SCORE = {num:.2f} / {den:.2f} = {pct:.0f}%")
    return pct

if os.path.exists(RESUME_PDF):
    score_skills(jd_skills)

# %% 8. Hand labels: show, set_gold, evaluate
def same(a, b):
    a, b = norm(a), norm(b)
    return a == b or (len(a) >= 3 and len(b) >= 3 and (f" {a} " in f" {b} " or f" {b} " in f" {a} "))

posts = json.load(open(LABELS, encoding="utf-8")) if os.path.exists(LABELS) else []

def show(i):
    p = posts[i]
    print(f"[{i}] {p['key']}   checked={p['checked']}\n")
    for n, s in enumerate(split_sentences(p["description"])):
        print(f"{n:3} | {s}")

def set_gold(i, skills_csv, split=None):
    posts[i]["skills"] = [x.strip().lower() for x in skills_csv.split(",") if x.strip()]
    posts[i]["checked"] = True
    if split:
        posts[i]["split"] = split
    json.dump(posts, open(LABELS, "w", encoding="utf-8"), indent=2, ensure_ascii=False)
    print(f"saved {len(posts[i]['skills'])} skills for post {i}")

def evaluate(only_checked=False, split=None):
    ex = tp_n = gold_found = gold_n = 0
    for p in posts:
        if only_checked and not p["checked"]:
            continue
        if split and p.get("split") != split:
            continue
        pred = [f["text"] for f in pipeline_skills(p["description"]).values()]
        gold = p["skills"]
        tp = [t for t in pred if any(same(t, g) for g in gold)]
        fp = [t for t in pred if t not in tp]
        fn = [g for g in gold if not any(same(t, g) for t in pred)]
        ex += len(pred); tp_n += len(tp); gold_found += len(gold) - len(fn); gold_n += len(gold)
        print(f"\n{p['key'][:60]}   checked={p['checked']}")
        print(f"  extracted {len(pred)}  gold {len(gold)}  "
              f"precision {len(tp)/max(len(pred),1):.2f}  recall {(len(gold)-len(fn))/max(len(gold),1):.2f}")
        print("  WRONG :", fp)
        print("  MISSED:", fn)
    print(f"\nOVERALL  precision {tp_n/max(ex,1):.2f}   recall {gold_found/max(gold_n,1):.2f}   "
          f"({ex} extracted, {gold_n} gold)")
    print("Trust these numbers only when every post is checked=True.")

if posts:
    evaluate()
else:
    print("No labels file yet. Copy samples/labels_draft.json to labels.json and set LABELS in cell 0.")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

labels: {0: 'B', 1: 'I', 2: 'O'}
skill                                source     kind    section           nice  alias
model building                       knowledge  domain  responsibilities  False 
Python                               knowledge  tool    requirements      False 
data science libraries               knowledge  domain  requirements      False 
NumPy                                knowledge  domain  requirements      False 
Pandas                               knowledge  domain  requirements      False 
Scikit                               knowledge  domain  requirements      False 
learn                                knowledge  domain  requirements      False 
SQL                                  knowledge  tool    requirements      False 
data extraction and manipulation     knowledge  domain  requirements      False 
dashboard creation                   knowledge  domain  requirements      False 
PowerBI                              knowledge  domain  requirements   

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

job skill                          w     evidence       credit
model building                     0.4   meaning        0.25
      near: Data Modelling                 0.58   <- Skills section: Data Modelling
      near: Model Validation               0.45   <- Skills section: Model Validation
      near: Feature Engineering            0.38   <- Skills section: Feature Engineering
Python                             1.0   listed/found   1.0
data science libraries             0.4   meaning        0.5
      near: Data Science & Engineering     0.7   <- M.Tech in Data Science & Engineering from BITS Pilani with 4+ years of
      near: Databricks                     0.55   <- Skills section: Databricks
      near: Business Intelligence          0.49   <- Skills section: Business Intelligence
NumPy                              0.4   listed/found   1.0
Pandas                             0.4   listed/found   1.0
Scikit                             0.4   text mention   1.0
learn                  